In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import json

# Load your model and tokenizer (modify based on your model)
MODEL_PATH = "jonday/wuc-model"  # Update this path
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

with open("wuc_mapping.json", "r") as file:
    wuc_mapping = json.load(file)
with open("codes.json", "r") as file:
    wuc_defs = json.load(file)
with open("main_system.json", "r") as file:
    main_system = json.load(file)
index_to_wuc = {v: k for k, v in wuc_mapping.items()}

# Ensure model is in evaluation mode
model.eval()

model.safetensors:  52%|#####2    | 231M/443M [00:00<?, ?B/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# Example batch of texts
texts = ["First input sentence", "Second input sentence", "Third input sentence"]

# Tokenize in batch
inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True)

# Run inference without computing gradients
with torch.no_grad():
    outputs = model(**inputs)

# Compute probabilities and predictions in batch
probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)  # Get probabilities
predicted_classes = torch.argmax(outputs.logits, dim=1)  # Get predicted indices
confidences = probabilities.gather(1, predicted_classes.unsqueeze(1)).squeeze().tolist()  # Get confidence scores

# Convert predictions back to WUCs and definitions
batch_results = []
for i, predicted_class in enumerate(predicted_classes.tolist()):
    wuc = index_to_wuc.get(predicted_class, "Unknown WUC")
    definition = wuc_defs.get(wuc, "Unknown Definition")
    system = main_system.get(wuc[:2], "Unknown Main System")
    
    batch_results.append({
        "text": texts[i],
        "predicted_class": predicted_class,
        "wuc": wuc,
        "definition": definition,
        "system": system,
        "confidence": confidences[i] * 100  # Convert to percentage
    })

# Print batch results
for result in batch_results:
    print(result)